# 📝 텍스트 임베딩 과제 LV2(응용) — 의미 검색·군집 평가·이미지 임베딩

> 이 과제는 교안에서 익힌 **임베딩·코사인 유사도·KMeans 군집·엘보우·실루엣·TF-IDF 키워드·CLIP 이미지 임베딩** 을 **서로 조합**해 한 단계 더 나아갑니다. 도메인은 **상품 리뷰**(전자제품·화장품·식품)와 **실제 사진**(고양이·자동차·배)입니다.

## 풀이 방법
1. 맨 위 **제공 코드 셀**(라이브러리·모델 로드·데이터 준비)을 먼저 위에서부터 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
4. 막히면 `힌트` 를 펼쳐 보세요.

> **자가채점이 없는 문제**: 그래프 문제(10)입니다. 정답 노트북의 완성 그래프와 비교하세요.

- 리뷰 데이터는 `data/product_reviews.csv`(18개, 열: `review` 리뷰 본문, `category` 카테고리(전자제품/화장품/식품)) 를 씁니다.
- 사진은 `data/photos/{cat,car,ship}/` 아래 카테고리별 5장(총 15장)입니다.
- 군집 문제(3~6, 10)는 제공 셀이 만들어 둔 **2차원 좌표 `coords`** 를 사용합니다(768차원 원본보다 군집 구조가 또렷해요 — 실루엣 값의 크기보다 **어느 k가 가장 큰지**로 판단합니다).

화이팅!

아래 셀들을 먼저 실행해 라이브러리·모델·데이터를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from umap import UMAP
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

이 과제는 **텍스트 임베딩 모델**(문장→768차원)과 **이미지 임베딩 모델 CLIP**(사진→512차원) 을 모두 씁니다. 두 모델을 미리 불러옵니다(처음 실행 시 다운로드로 시간이 걸릴 수 있어요).

In [ ]:
# [제공 코드] 텍스트·이미지 임베딩 모델을 모두 불러옵니다.
text_model = SentenceTransformer('jhgan/ko-sroberta-multitask')   # 한국어 문장 → 768차원
clip_model = SentenceTransformer('clip-ViT-B-32')                 # 이미지/영어텍스트 → 512차원

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
새 데이터셋이니 분석 전에 먼저 파악합니다. `head()`(앞부분)·`info()`(열·자료형·결측)로 구조를 보고, **카테고리 분포**도 확인합니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 상품 리뷰 데이터를 먼저 살펴봅니다.
reviews = pd.read_csv('data/product_reviews.csv')
print("행·열 크기:", reviews.shape)
print("\n[앞 5행] head()"); display(reviews.head())
print("\n[열·자료형·결측] info()"); reviews.info()
print("\n[카테고리 분포] value_counts()"); display(reviews["category"].value_counts())

## 좌표 준비 — 리뷰 임베딩과 2차원 요약
리뷰 18개를 임베딩해 `review_emb`(18×768) 에 담고, 교안에서 배운 **UMAP** 으로 이를 **2차원 좌표 `coords`(18×2)** 로 요약합니다. 원본 768차원은 카테고리 안에서도 표현이 다양해 군집 구조가 흐릿한데, 2차원으로 요약하면 세 카테고리 구조가 또렷해집니다. **군집 문제(3~6, 10)는 이 `coords` 를 사용**하고, 의미 검색 문제(1~2)는 `review_emb`(원본 임베딩)를 씁니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 리뷰를 임베딩하고 UMAP으로 2차원 좌표를 만듭니다.
review_emb = text_model.encode(reviews['review'].tolist())
coords = UMAP(n_components=2, n_neighbors=5, min_dist=0.05,
              random_state=0).fit_transform(review_emb)
print("리뷰 임베딩 shape =", review_emb.shape, "  (리뷰 18개 × 768차원)")
print("2차원 좌표 shape =", coords.shape, "  (리뷰 18개 × 2차원)")

## 1. 의미 검색 함수 만들기 — 임베딩 + 코사인 + 정렬 (조합)
**배경**: 검색창에 문장을 넣으면 **뜻이 비슷한 리뷰**를 찾아 주는 기능을 만듭니다. 키워드가 정확히 겹치지 않아도(예: "휴대폰"↔"배터리") 임베딩·코사인 유사도로 **의미가 가까운** 리뷰를 찾는 것이 핵심입니다. **쿼리 임베딩 + 코사인 유사도 + 정렬**을 하나의 함수로 조합합니다.

**요구사항**:
- 함수 `search_reviews(query, k=3)` 를 정의하세요. 쿼리 문자열과 정수 `k` 를 받아, 쿼리와 가장 비슷한 리뷰 **k개의 행 인덱스**를 **유사도가 높은 순서로** 담은 리스트를 반환합니다.
- 함수 안에서 쿼리를 임베딩(`text_model.encode`)하고, `review_emb` 의 18개 리뷰와의 코사인 유사도를 구한 뒤, **유사도가 큰 순서(내림차순)로 상위 `k` 개의 행 인덱스**를 골라 리스트로 반환하세요(`argsort` 는 오름차순이라 결과를 뒤집어야 합니다).
- 정의한 함수를 `query = '휴대폰 배터리가 오래가는지 궁금해요'` 로 호출해 결과 인덱스를 `result` 에 담고, 그 리뷰들을 출력해 보세요.

**예시**
```
result = search_reviews('휴대폰 배터리가 오래가는지 궁금해요', 3)
len(result)                          →  3
reviews.loc[result[0], 'category']   →  '전자제품'   (가장 비슷한 리뷰는 전자제품)
reviews.loc[result[0], 'review']     →  '배터리'가 들어간 리뷰
```
<details><summary>힌트</summary>

```text
접근방법:
- 쿼리 한 문장을 임베딩하고, 미리 만들어 둔 리뷰 임베딩과의 코사인 유사도를 구해 큰 순서로 정렬한다.

세부구현:
1. 함수 search_reviews(query, k=3) 를 def 로 정의한다
2. text_model.encode([query]) 로 쿼리를 (1, 768) 임베딩으로 만든다
3. 쿼리 임베딩과 review_emb 의 코사인 유사도 배열(첫 행)이 18개 리뷰와의 유사도다
4. 유사도를 내림차순 정렬한 인덱스에서 앞의 k개를 골라 리스트로 반환한다(argsort 는 오름차순이라 뒤집기)
5. 함수를 예시 쿼리로 호출해 result 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert callable(search_reviews)
assert len(result) == 3
assert reviews.loc[result[0], 'category'] == '전자제품'
assert '배터리' in reviews.loc[result[0], 'review']
print("✅ 문제1 통과!")

## 2. 여러 쿼리의 카테고리 판별 (짝 문제 — 판별형)
**배경**: 문제 1의 검색 아이디어를 **분류**에 응용합니다. 새 쿼리가 들어오면 **가장 비슷한 리뷰의 카테고리**로 그 쿼리의 주제를 추측할 수 있습니다(최근접 이웃 분류의 기본형). 세 개의 쿼리가 각각 어느 카테고리로 판별되는지 확인합니다.

**요구사항**:
- 아래 세 쿼리를 리스트 `queries` 로 두세요.
```
queries = ['무선 이어폰 노이즈 캔슬링 성능이 좋은 제품',
           '백탁 없이 촉촉하게 발리는 선크림',
           '아침에 내려 마실 신선한 커피 원두']
```
- 각 쿼리에 대해 **가장 비슷한 리뷰 1개의 카테고리**를 구해, 세 카테고리를 순서대로 담은 리스트 `pred_categories` 를 만드세요. (문제 1의 `search_reviews(q, 1)` 를 재사용하면 첫 번째 인덱스의 `category` 로 알 수 있습니다.)

**예시**
```
pred_categories  →  ['전자제품', '화장품', '식품']
```
<details><summary>힌트</summary>

```text
접근방법:
- 각 쿼리마다 top1 리뷰를 찾아 그 리뷰의 카테고리를 모은다.

세부구현:
1. queries 리스트를 만든다
2. 빈 리스트 pred_categories 를 만들고 각 쿼리를 반복한다
3. search_reviews(query, 1) 의 첫 인덱스로 reviews 의 category 를 읽어 append 한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert pred_categories == ['전자제품', '화장품', '식품']
print("✅ 문제2 통과!")

## 3. 엘보우 방법 — k별 관성으로 팔꿈치 찾기 (조합)
**배경**: 군집을 몇 개로 나눌지(k)는 정답이 없어 **여러 k 를 시도해 비교**합니다. **반복문 + 관성(inertia)** 을 조합해, k=1부터 7까지 각각의 관성을 구하고 그래프가 팔을 굽히는 **팔꿈치** 지점을 찾습니다. 관성은 각 점이 자기 군집 중심에서 떨어진 **제곱 거리의 합**으로, k 가 커질수록 항상 줄어듭니다.

**요구사항**:
- 제공된 **2차원 좌표 `coords`** 를 사용합니다.
- k 를 1~7 로 바꿔 가며 `KMeans(n_clusters=k, random_state=0, n_init=10).fit(coords)` 로 군집을 만들고 `.inertia_` 를 구해, k=1..7 순서의 리스트 `inertias`(길이 7)에 담으세요.
- 이웃한 k 사이의 **관성 감소량**을 살펴, 감소가 확 작아지기 **직전**의 k(팔꿈치)를 아래 서술 셀에 적으세요.

**예시**
```
len(inertias)  →  7
inertias 는 계속 줄어들지만, 어느 k 이후로는 감소폭이 급격히 작아진다(팔꿈치).
```
<details><summary>힌트</summary>

```text
접근방법:
- k 를 1부터 7까지 돌며 각 KMeans 의 관성을 리스트에 모으고, 감소폭이 꺾이는 지점을 눈으로 찾는다.

세부구현:
1. 빈 리스트 inertias 를 만든다
2. k 를 1~7 로 돌며 KMeans(n_clusters=k, random_state=0, n_init=10) 를 coords 에 fit 한다
3. 그 모델의 inertia_ 를 inertias 에 append 한다
4. 이웃 k 의 관성 차이를 출력해 감소폭이 급격히 작아지는 직전 k 를 팔꿈치로 읽는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(inertias) == 7
assert all(inertias[i] < inertias[i - 1] for i in range(1, 7))   # k가 커지면 관성은 계속 감소
# coords(2차원) 로 풀었는지 — 768차원 emb 로 풀면 관성 규모가 전혀 다르다
assert 40 < inertias[0] < 90, 'coords(2차원)가 아니라 다른 데이터로 풀었을 수 있어요'
# 팔꿈치가 k=3 인지 — 2->3 감소량이 3->4 감소량보다 훨씬 크다
assert (inertias[1] - inertias[2]) > 10 * (inertias[2] - inertias[3])
print("✅ 문제3 통과!")

*(팔꿈치 지점 k 를 여기에 적으세요)*

## 4. 실루엣로 최적의 k 고르기 (조합)
**배경**: 엘보우가 애매할 때 **실루엣 계수**로 군집 품질을 정량 비교합니다(−1~1, 클수록 잘 뭉침). **반복문 + 실루엣**을 조합해 여러 k 의 실루엣을 구하고, 가장 높은 k 를 최적값으로 고릅니다.

**요구사항**:
- 제공된 **2차원 좌표 `coords`** 를 사용합니다.
- k 를 2~5 로 바꿔 가며 `KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(coords)` 로 군집 라벨을 만들고 `silhouette_score(coords, labels)` 를 구해, `{k: 실루엣}` 형태의 딕셔너리 `sil_scores` 에 담으세요.
- `sil_scores` 에서 실루엣이 가장 높은 k 를 정수 `best_k` 에 담으세요.

**예시**
```
sil_scores  →  {2: 0.594, 3: 0.673, 4: 0.570, 5: 0.532}   (하드웨어에 따라 조금 다를 수 있음)
best_k      →  3
```
<details><summary>힌트</summary>

```text
접근방법:
- 2~5 범위를 돌며 각 k 의 KMeans 실루엣을 딕셔너리에 모으고, 값이 가장 큰 키를 고른다.
- 딕셔너리에서 '값이 가장 큰 키'는 max 에 key 를 지정해 찾는다.

세부구현:
1. 빈 딕셔너리 sil_scores 를 만든다
2. k 를 2~5 로 돌며 KMeans(random_state=0, n_init=10) 라벨을 만들고 실루엣을 sil_scores[k] 에 담는다
3. sil_scores 에서 값이 가장 큰 키를 best_k 에 담는다(max 에 key 로 sil_scores.get 지정)
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(sil_scores.keys()) == {2, 3, 4, 5}
assert best_k == 3
print("✅ 문제4 통과!")

## 5. 최적 k의 실루엣 점수 (짝 문제 — 값 확인형)
**배경**: 문제 4에서 고른 **최적 k(=3)** 의 군집이 실제로 얼마나 잘 뭉쳤는지, 실루엣 점수 하나로 확인합니다. 같은 실루엣 개념을 이번엔 **특정 k 의 점수 값**으로 봅니다.

**요구사항**:
- 문제 4의 `sil_scores` 에서 **k=3 의 실루엣 점수**를 꺼내 **소수 첫째 자리로 반올림**해 `score_k3` 에 담으세요. (값을 직접 적지 말고 `sil_scores` 에서 꺼내 계산해야 합니다.)

**예시** — 형식만 보여 줍니다(실제 값은 직접 구하세요)
```
score_k3  →  0.x   (소수 첫째 자리 실수 하나)
```
<details><summary>힌트</summary>

```text
접근방법:
- 이미 만든 실루엣 딕셔너리에서 k=3 값을 꺼내 소수 첫째 자리로 반올림한다.

세부구현:
1. 딕셔너리에서 키 3 의 값을 꺼내, 소수 첫째 자리로 반올림해 score_k3 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert score_k3 == round(sil_scores[3], 1)   # sil_scores 에서 꺼내 반올림했는지
assert 0.6 <= score_k3 <= 0.8   # 반올림한 한 자리 값(하드웨어에 따라 조금 달라질 수 있음)
assert sil_scores[3] == max(sil_scores.values())   # k=3이 실루엣 최댓값
print("✅ 문제5 통과!")

## 6. 군집 대표 키워드 — TF-IDF로 이름 붙이기 (조합)
**배경**: 군집 번호(0·1·2)만으로는 무슨 묶음인지 알 수 없습니다. **KMeans 군집 + TF-IDF 키워드 추출**을 조합해, 각 군집을 대표하는 단어를 뽑고 그것이 실제 카테고리와 맞물리는지 대조합니다.

**요구사항**:
- 제공된 **2차원 좌표 `coords`** 에 `KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(coords)` 로 군집 라벨을 만들어 `km_labels` 에 담으세요.
- 각 군집(0·1·2)에 속한 리뷰들을 **하나의 문서로 합쳐** 리스트 `cluster_docs`(3개)에 담으세요(`' '.join(...)`).
- `TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')` 로 세 문서를 벡터화하고, 각 군집에서 TF-IDF 점수 **상위 5개 단어**를 뽑아 `{군집번호: [단어5개]}` 형태의 딕셔너리 `cluster_keywords` 에 담으세요.
- 군집이 실제 카테고리와 얼마나 맞물리는지 `pd.crosstab(km_labels, reviews['category'])` 를 `ct` 에 담아 확인하세요.

**예시**
```
cluster_keywords  →  {0: [...5단어...], 1: [...], 2: [...]}
ct                →  각 군집이 한 카테고리에만 6개씩 몰린 완벽한 대각선
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 좌표를 3개 군집으로 나누고, 군집별 리뷰를 합쳐 TF-IDF 로 특징 단어를 뽑는다.
- TF-IDF 는 그 군집에 자주·다른 군집엔 드문 단어에 높은 점수를 준다.

세부구현:
1. KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(coords) 로 km_labels 를 만든다
2. 군집 0·1·2 마다 그 군집 리뷰를 ' '.join 으로 합쳐 cluster_docs(3개)에 담는다
3. TfidfVectorizer 로 cluster_docs 를 fit_transform 하고 get_feature_names_out 으로 단어 목록을 얻는다
4. 각 군집 행의 TF-IDF 점수를 내림차순 정렬해 상위 5개 인덱스를 골라, feature 이름으로 단어를 얻어 cluster_keywords 에 담는다
5. pd.crosstab 으로 군집×카테고리 교차표 ct 를 만든다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(cluster_keywords) == 3
assert all(len(words) == 5 for words in cluster_keywords.values())
assert int(ct.max(axis=1).sum()) == 18   # 각 군집이 한 카테고리로 완벽히 갈림(6+6+6)
print("✅ 문제6 통과!")

## 7. 이미지 임베딩 — 같은/다른 카테고리 코사인 비교 (조합)
**배경**: 임베딩은 텍스트만의 것이 아닙니다. **CLIP** 으로 사진도 벡터로 바꾸면, 같은 원리가 통합니다 — "같은 카테고리 사진끼리 더 비슷하다". **이미지 임베딩 + 코사인 유사도 집계**를 조합해 이를 수치로 확인합니다. `data/photos/` 에는 고양이·자동차·배 사진이 카테고리별 5장씩(총 15장) 있습니다.

**요구사항**:
- 카테고리 순서 `['cat', 'car', 'ship']` 대로 15장의 경로와 카테고리 라벨을 모아 `image_paths`·`image_cats` 를 만들고, `Image.open` 으로 연 사진들을 `clip_model.encode(...)` 로 임베딩해 `image_emb`(15×512) 에 담으세요.
- `cosine_similarity(image_emb)` 로 15×15 유사도 행렬 `image_sim` 을 만드세요.
- 서로 다른 두 사진 쌍을 훑어, **같은 카테고리** 쌍의 코사인 평균을 `intra_mean`, **다른 카테고리** 쌍의 평균을 `inter_mean` 에 담으세요.

**예시**
```
image_emb.shape  →  (15, 512)
intra_mean       →  약 0.71   (같은 카테고리끼리가 더 비슷)
inter_mean       →  약 0.51
intra_mean > inter_mean  →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- 15장을 카테고리 순서대로 임베딩하고, 모든 사진 쌍의 코사인 유사도를 같은/다른 카테고리로 나눠 평균낸다.

세부구현:
1. ['cat','car','ship'] 마다 1~5 번 사진 경로를 image_paths 에, 카테고리를 image_cats 에 담는다
   (경로 형식: data/photos/<카테고리>/<카테고리>_<번호>.jpg  예: data/photos/cat/cat_1.jpg)
2. Image.open 으로 연 사진들을 clip_model.encode 로 image_emb 에 담는다
3. cosine_similarity(image_emb) 로 15x15 유사도 image_sim 을 만든다
4. a<b 인 쌍을 돌며 image_cats[a]==image_cats[b] 면 intra, 아니면 inter 리스트에 유사도를 모은다
5. 두 리스트의 평균을 intra_mean·inter_mean 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert image_emb.shape == (15, 512)
assert intra_mean > inter_mean          # 같은 카테고리끼리가 더 비슷
assert intra_mean > 0.7                  # 같은 카테고리 유사도가 충분히 높다
print("✅ 문제7 통과!")

## 8. 쿼리 이미지와 가장 비슷한 사진 (짝 문제 — 최근접 검색)
**배경**: 문제 7의 유사도를 **검색**에 응용합니다. 한 장의 **쿼리 사진**을 기준으로, 자기 자신을 뺀 나머지 중 **가장 비슷한 사진**을 찾으면 "이 사진과 닮은 사진" 검색이 됩니다. 같은 카테고리 사진이 뽑히는지 확인합니다.

**요구사항**:
- 문제 7의 `image_sim`·`image_cats` 를 사용합니다.
- 쿼리 사진 인덱스를 `target = 10`(배 사진 중 하나) 으로 두세요.
- `image_sim[target]` 에서 **자기 자신(target)을 제외**하고 유사도가 가장 큰 사진의 인덱스를 `nearest` 에 담으세요(자기 자신은 유사도 1 이라 반드시 빼야 합니다).
- 쿼리와 최근접 사진의 카테고리를 비교해, 같은 카테고리인지 `same_category`(True/False)에 담으세요.

**예시**
```
target = 10          # 'ship'
nearest              →  같은 'ship' 사진의 인덱스
same_category        →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- 쿼리 행의 유사도에서 자기 자신을 -1 로 지우고 argmax 로 가장 비슷한 사진을 찾는다.

세부구현:
1. target = 10 으로 둔다
2. 그 사진의 유사도 행을 **복사**한 뒤(원본을 훼손하지 않도록), 자기 자신 자리의 값을 아주 작은 값으로 덮어 후보에서 뺀다
3. 남은 값 중 가장 큰 곳의 위치를 찾아 nearest 에 담는다
4. image_cats[target] 와 image_cats[nearest] 가 같은지 same_category 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert nearest != target
assert bool(same_category) == True   # 최근접 사진은 같은 카테고리(배)
# 실제로 유사도가 가장 큰 사진을 골랐는지 확인(값을 지어내면 걸린다)
assert image_cats[nearest] == image_cats[target]
assert nearest == int(np.delete(np.arange(len(image_cats)), target)[
    np.delete(image_sim[target], target).argmax()])
print("✅ 문제8 통과!")

## 9. 텍스트로 사진 찾기 — 멀티모달 검색 (조합)
**배경**: CLIP 의 진짜 힘은 **글과 사진을 같은 공간**에 담는 것입니다. 그래서 **글 쿼리로 사진을 검색**할 수 있어요 — 교안에서 본 멀티모달입니다. 문제 7의 `image_emb`·`image_cats` 를 그대로 쓰고, **영어 설명 문장**을 임베딩해 가장 비슷한 사진을 찾습니다(CLIP 은 영어로 학습돼 영어 쿼리가 더 정확합니다).

**요구사항**:
- 쿼리 리스트 `image_queries = ['a photo of a cat', 'a photo of a car', 'a photo of a ship']` 를 CLIP 으로 (**문제 2 의 `queries` 와 이름이 겹치지 않도록 `image_queries` 를 씁니다**) 임베딩해 `query_emb`(3×512)에 담으세요.
- 쿼리 임베딩과 문제 7의 `image_emb` 로 **3×15 코사인 유사도 행렬**을 만들고, 각 쿼리(행)마다 **가장 비슷한 사진의 카테고리**를 순서대로 리스트 `predicted`(길이 3)에 담으세요.
- 세 글 쿼리가 각각 고양이·자동차·배 사진을 찾아, `predicted` 가 `['cat', 'car', 'ship']` 이 되어야 합니다.

**예시**
```
query_emb.shape  →  (3, 512)
predicted        →  ['cat', 'car', 'ship']   (글이 알맞은 사진을 찾음)
```
<details><summary>힌트</summary>

```text
접근방법:
- 영어 쿼리들을 임베딩해 15장과의 코사인을 구하고, 쿼리마다 가장 비슷한 사진의 카테고리를 모은다.

세부구현:
1. image_queries 를 clip_model 로 임베딩해 query_emb 에 담는다
2. 쿼리 임베딩과 image_emb 의 코사인 유사도로 3x15 행렬을 만든다
3. 각 행에서 가장 큰 값의 사진 인덱스를 찾는다(행마다 argmax)
4. 그 인덱스의 image_cats 값을 순서대로 predicted 에 모은다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert query_emb.shape == (3, 512)
assert predicted == ['cat', 'car', 'ship']
print("✅ 문제9 통과!")

## 10. 군집 산점도 그리기 (그래프 — 스크린샷 재현)
**배경**: 2차원 좌표 `coords` 를 **군집별 색으로** 그리면, 세 카테고리가 눈으로도 갈라지는지 확인할 수 있습니다. **군집 결과 + 산점도 시각화**를 조합합니다.

**요구사항**:
- 문제 6의 `km_labels`(k=3 군집 라벨)를 사용합니다(없으면 다시 만드세요).
- `plt.figure(figsize=(7, 5.5))` 로 새 그림을 연 뒤, 군집 0·1·2 마다 그 군집의 점만 골라 `plt.scatter(coords[군집마스크, 0], coords[군집마스크, 1], ...)` 로 서로 다른 색으로 그리고 `label='군집 i'` 를 주세요.
- 범례(`plt.legend()`)·제목·축 이름(`UMAP 1`, `UMAP 2`)을 달고 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프처럼 세 군집이 서로 다른 색의 세 덩어리로 나뉘면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 군집 번호마다 그 군집의 점만 골라 색을 달리해 산점도를 겹쳐 그린다.

세부구현:
1. km_labels 가 없으면 KMeans(n_clusters=3, random_state=0, n_init=10).fit_predict(coords) 로 만든다
2. plt.figure 로 새 그림을 연다
3. i 를 0·1·2 로 돌며 (km_labels == i) 마스크로 그 군집 점만 scatter 로 그린다(label='군집 i')
4. 범례·제목·축 이름을 달고 plt.show 로 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv2_q10_cluster_scatter.png" width="560">

In [ ]:
# 여기에 코드를 작성하세요

## 11. 모델 고르기 — 최대 입력 길이 함정을 직접 재현하기 (조합)
**배경**: 교안 6절에서 모델을 고를 때 **가장 자주 놓치는 것**이 `max_seq_length` 라고 배웠습니다. 이 모델은 **128토큰**까지만 읽고 나머지를 **에러도 경고도 없이 버립니다**. 말로만 듣지 말고 **직접 재현해** 그 위험을 몸으로 확인하세요.

**요구사항**:
- 뒤쪽만 정반대인 **긴** 문장 두 개를 만듭니다 — 같은 앞부분을 아주 길게 반복한 뒤(`'무선 이어폰 배터리 성능이 좋아요 ' * 60` 처럼 128토큰을 훌쩍 넘기게), 한쪽엔 `'결론적으로 최악입니다'`, 다른 쪽엔 `'결론적으로 최고입니다'` 를 **맨 뒤에** 붙이세요.
- 두 문장을 임베딩해 코사인 유사도를 `sim_long` 에 담으세요. **뒤가 잘려 1.0 에 가깝게** 나옵니다.
- 같은 앞부분을 **아주 짧게**(`* 2`) 줄여 같은 뒷문구를 붙인 두 문장의 유사도를 `sim_short` 에 담으세요.
- 두 값을 나란히 출력해 **얼마나 차이 나는지** 확인하세요.

**예시**
```
sim_long   ->  1.0 에 아주 가까움 (뒤가 잘려 두 문장이 사실상 같은 벡터)
sim_short  ->  뚜렷하게 더 낮음   (안 잘려서 최악 vs 최고가 구분됨)
```
<details><summary>힌트</summary>

```text
접근방법:
- 같은 앞부분을 길게/짧게 두 번 만들어, 각각 정반대 뒷문구를 붙인 문장쌍의 유사도를 비교한다.

세부구현:
1. 앞부분 문자열을 곱셈으로 길게 만든다(긴 것 60회, 짧은 것 2회)
2. 각 앞부분에 '결론적으로 최악입니다' / '결론적으로 최고입니다' 를 이어 붙여 2개짜리 리스트를 만든다
3. 리스트를 인코딩해 두 벡터의 코사인 유사도를 구한다(교안 3절과 같은 방식)
4. 긴 쪽을 sim_long, 짧은 쪽을 sim_short 에 담아 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sim_long > 0.99, '앞부분을 더 길게 만들어 128토큰을 넘기세요'
assert sim_short < sim_long - 0.1, '짧은 쪽은 뒷문구가 살아 유사도가 뚜렷이 낮아야 합니다'
print("✅ 문제11 통과! 긴 글은 경고 없이 잘린다 - 모델을 고를 때 max_seq_length 를 꼭 확인하자")

## 참고 — `n_neighbors` 를 바꿔 보기 (무채점)
UMAP 의 **`n_neighbors`** 는 "각 점의 이웃을 몇 명까지 볼 것인가"입니다. **작게** 주면 가까운 몇 개만 보고 **작은 덩어리들이 또렷하게** 갈라지고, **크게** 주면 전체 모양을 함께 보느라 **덩어리가 느슨하게** 풀립니다. 아래를 실행해 문제 10의 그림(`n_neighbors=5`)과 **모양이 어떻게 달라지는지** 눈으로 비교해 보세요. 채점은 없습니다.

> 같은 데이터·같은 도구인데 **그림이 달라진다**는 점이 핵심입니다 — 2차원 그림은 **하나의 요약**이지 정답 지도가 아닙니다.

In [ ]:
# [제공 코드] 같은 임베딩을 n_neighbors 만 바꿔 두 장으로 그려 비교합니다.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, nn in zip(axes, [2, 12]):
    alt = UMAP(n_components=2, n_neighbors=nn, min_dist=0.05,
               random_state=0).fit_transform(review_emb)
    for cat in reviews['category'].unique():
        mask = (reviews['category'] == cat).values
        ax.scatter(alt[mask, 0], alt[mask, 1], s=80, label=cat, alpha=0.8)
    ax.set_title(f'UMAP n_neighbors={nn}')
    ax.legend()
plt.tight_layout(); plt.show()

## 12. 표준화가 군집을 바꾼다 — 생선 데이터 (조합)
**배경**: 교안 1-2 절에서 **축마다 단위가 다르면 큰 숫자 축이 거리를 독차지**한다고 배웠습니다. 이번엔 **여러분이 직접** 표준화 전후를 비교해, 숫자로 확인하세요.

**요구사항**:
- `data/fish.csv` 를 읽어 `fish` 에 담고, 특성 5개(`Weight`·`Length`·`Diagonal`·`Height`·`Width`)만 골라 `X` 에 담으세요(`Species` 는 정답 라벨이라 제외).
- `StandardScaler` 로 표준화한 배열을 `X_scaled` 에 담으세요.
- **원본 `X`** 와 **표준화한 `X_scaled`** 각각에 `KMeans(n_clusters=7, random_state=42, n_init=10)` 를 돌려, `Species` 와의 **ARI**(`adjusted_rand_score`)를 `ari_raw`·`ari_scaled` 에 담으세요.
- 두 값을 나란히 출력해 **어느 쪽이 실제 종에 가까운지** 확인하세요.

**예시**
```
ari_raw     →  약 0.13   (무게가 거리를 독차지해 종 구분이 잘 안 됨)
ari_scaled  →  약 0.30   (다섯 특성이 고르게 반영돼 실제 종에 가까워짐)
```
<details><summary>힌트</summary>

```text
접근방법:
- 같은 KMeans 를 원본과 표준화 배열에 각각 돌려 ARI 를 비교한다.

세부구현:
1. pd.read_csv 로 fish.csv 를 읽고, 특성 5개 열만 .values 로 뽑아 X 에 담는다
2. StandardScaler().fit_transform(X) 로 X_scaled 를 만든다
3. KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict 를 각각 돌린다
4. adjusted_rand_score(fish['Species'], 라벨) 로 ari_raw·ari_scaled 를 구해 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert X.shape == (159, 5), 'Species 를 뺀 특성 5개만 골라야 합니다'
# 표준화가 실제로 됐는지 — 평균 0, 표준편차 1
assert abs(X_scaled.mean()) < 1e-9 and abs(X_scaled.std() - 1) < 1e-9
# 핵심: 표준화한 쪽이 실제 종과 더 잘 맞아야 한다
assert ari_scaled > ari_raw, '표준화한 쪽의 ARI 가 더 높아야 합니다 — 두 배열을 바꿔 넣지 않았는지 확인하세요'
assert 0.25 < ari_scaled < 0.40
print('✅ 통과!')